# 第6章　实战：用 CNN 识别手写数字（MNIST）

做图像识别的经典 **CNN（卷积神经网络）**，分类手写数字 0〜9。
把目前学的全部（张量・自动微分・训练循环・nn.Module・DataLoader）整合到一条流水线。

本章目标：搭 CNN 在 MNIST 上达到约 98% 准确率，并了解 GPU 用法。

> **本笔记使用方法**：从上到下 `Shift + Enter`。多数章节不需要 GPU；较重的章节会说明。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 6-0. 启用 GPU（Colab）
这章较重，推荐 GPU。**菜单「代码执行程序」→「更改运行时类型」→ GPU (T4)**，再重跑第一个单元格。
没有 GPU 也能用 CPU 跑（只是慢些，可减少 epoch）。

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 6-1. 图像张量的形状与"卷积"的直觉

- 图像批量的形状是 **`(N, C, H, W)`** =（张数, 通道, 高, 宽）。MNIST 是黑白，所以 `C=1`。
- **卷积 (`nn.Conv2d`)**：让一个小滤波器在图像上滑动，检测"边缘""曲线"等**局部图案**。
- **池化 (`nn.MaxPool2d`)**：缩小图像、概括信息（对位置偏移更鲁棒、计算变少）。

先看 Conv 如何改变形状。

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
dummy = torch.randn(4, 1, 28, 28)        # 4张28x28黑白
out = conv(dummy)
print("输入:", dummy.shape, "-> 卷积后:", out.shape)   # (4, 8, 28, 28)
print("池化后:", nn.MaxPool2d(2)(out).shape)           # (4, 8, 14, 14)

## 6-2. 准备数据

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)
print("train:", len(train_ds), " test:", len(test_ds))

## 6-3. 定义 CNN 模型
`Conv → ReLU → Pool` 做两遍 → 展平(`Flatten`) → 全连接层输出到10类。

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28->14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14->7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),              # (N,32,7,7) -> (N, 32*7*7)
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),        # 10类（数字0〜9）
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN().to(device)      # 模型送到 GPU/CPU
print(model)

## 6-4. 训练
要点：**每个批量都 `xb, yb = xb.to(device), yb.to(device)`** 把数据也送到同一设备。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 3   # CPU 上设 1 也行。GPU 上 3〜5 可达 98%+
model.train()
for epoch in range(EPOCHS):
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)   # 数据也送到设备
        optimizer.zero_grad()        # ①
        logits = model(xb)           # ②
        loss = criterion(logits, yb) # ③
        loss.backward()              # ④
        optimizer.step()             # ⑤
        running += loss.item()
    print(f"epoch {epoch+1}: 平均loss = {running/len(train_loader):.4f}")

## 6-5. 用测试集计算准确率

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print(f"测试准确率: {100*correct/total:.2f}%")

## 6-6. 可视化预测（哪里错了？）

In [ ]:
import matplotlib.pyplot as plt
xb, yb = next(iter(test_loader))
with torch.no_grad():
    pred = model(xb.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(xb[i].squeeze(), cmap="gray")
    ok = pred[i].item() == yb[i].item()
    ax.set_title(f"pred {pred[i].item()} / true {yb[i].item()}",
                 color=("green" if ok else "red"), fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 练习 6
1. 增大 `EPOCHS`／改 `lr`，准确率怎么变？
2. 再加一层 Conv、改 `out_channels` 等改动结构（注意 `Flatten` 后的输入大小：形状变了要相应改 `nn.Linear` 的数）。
3. 把数据换成 **CIFAR-10**（彩色图，`C=3`）试试（`datasets.CIFAR10`、`Conv2d` 的 `in_channels=3`、按 32x32 算尺寸）。能感受到难度上升。

In [ ]:
# 在这里写你自己的代码并运行
